In [1]:
import pandas as pd
 
 
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/titanic.csv"
df = pd.read_csv(url)

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# Pregunta A

### ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?

La tasa de supervivencia fue del 38.8%

In [ ]:
vivos = df['Survived'].value_counts(normalize=True) * 100
print(vivos)

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64


# Pregunta B

### ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

El 74.2% de mujeres sobrevivieron, mientras que solo el 18.89% de hombres sobrevivieron

In [ ]:
vivosMujer = df.groupby('Sex')['Survived'].mean() * 100
print(vivosMujer)

Sex
female    74.203822
male      18.890815
Name: Survived, dtype: float64


# Pregunta C

### ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

La mayoria de estos pasajeros estaban en primera clase

In [ ]:
q1 = df['Fare'].quantile(0.25)  # 7.91
q3 = df['Fare'].quantile(0.75)  # 31.00

iqr = q3 - q1  # 23.09

limit_sup= q3 + 1.5 * iqr  # 65.63

# Filtrado de outliers
outliers_fare = df[df['Fare'] > limit_sup]
print(f'Cantidad de outliers: {len(outliers_fare)}')
print(outliers_fare['Pclass'].value_counts())

Cantidad de outliers: 116
Pclass
1    104
3      7
2      5
Name: count, dtype: int64


# Pregunta D

### ¿Qué significa que la media sea tan superior a la mediana?
La media es mayor porque los boletos caros de primera clase (outliers) inflan el promedio hacia arriba (asimetría a la derecha).

### Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?
Fare dominará por completo la distancia euclidiana debido a su escala tan alta (hasta 512), haciendo que el modelo ignore las demás variables (edad, clase, sexo) y aprenda con distorsiones causadas por los valores atípicos.

In [6]:
media_fare = df['Fare'].mean()
mediana_fare = df['Fare'].median()

print(f"Media de Fare: {media_fare:.2f}")
print(f"Mediana de Fare: {mediana_fare:.2f}")

Media de Fare: 32.20
Mediana de Fare: 14.45


# Pregunta E

### ¿Qué sesgo evitan al hacer esto?

Se evita el sesgo de muestreo y la distorsion de la distribucion previa de clases

In [ ]:
proporciones = df['Survived'].value_counts(normalize=True)

muestra_0 = df[df['Survived'] == 0].sample(n=int(round(proporciones[0] * 150)), random_state=42)
muestra_1 = df[df['Survived'] == 1].sample(n=int(round(proporciones[1] * 150)), random_state=42)

muestra_estratificada = pd.concat([muestra_0, muestra_1])

print("Proporción en la muestra estratificada:")
print(muestra_estratificada['Survived'].value_counts(normalize=True))
print(f"\nTamaño total de la muestra: {len(muestra_estratificada)}")

Proporción en la muestra estratificada:
Survived
0    0.613333
1    0.386667
Name: proportion, dtype: float64

Tamaño total de la muestra: 150


# Pregunta F

### ¿Qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

Se introduce el sesgo de supervivencia por datos faltantes que no son aleatorios

In [ ]:
promedio_edad_sobrevivientes = df.groupby('Survived')['Age'].mean()
print(promedio_edad_sobrevivientes)

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64


# Pregunta G

### ¿Deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo?

No se deberian de eliminar ya que estas tarifas extremas no son errores de captura, sino datos reales de pasajes VIP (los mas caretes)  
que aportan informacion valiosa sobre la alta probabilidad de supervivencia de la clase alta  
si se eliminan se podria degradar la capacidad del modelo para generalizar.

# Pregunta H

### ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

Porque el titulo es un buen predictor de la etapa de vida, lo que reduce la varianza de error al no asignarle la mediana general a un menor

In [ ]:
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
print("Medianas de edad por título:")
print(df.groupby('Title')['Age'].median())

Medianas de edad por título:
Title
Capt        70.0
Col         58.0
Countess    33.0
Don         40.0
Dr          46.5
Jonkheer    38.0
Lady        48.0
Major       48.5
Master       3.5
Miss        21.0
Mlle        24.0
Mme         24.0
Mr          30.0
Mrs         35.0
Ms          28.0
Rev         46.5
Sir         49.0
Name: Age, dtype: float64


# Pregunta I

### ¿Qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?
Si entrenamos a un algoritmo con 0.0, el modelo no podria aprender porque o todos los pasajeros sobrevivieron o todos murieron

In [ ]:
varianza_survived = df['Survived'].var()
print(f"Varianza de Survived: {varianza_survived:.4f}")

Varianza de Survived: 0.2368


# Pregunta J

### ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?

Ocurriria overfitting ya que cuando el modelo se entrena con caracteristicas muy especificas, el algoritmo memoriza el resultado exacto a ese unico caso

In [14]:
conteo_subgrupos = df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()
print(conteo_subgrupos)

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64
